# Panorama (MultiDiffusion) → Modular Diffusers — smoke (training-free ultra-wide)

Ultra-wide FLUX beyond its native aspect (MultiDiffusion, arXiv:2302.08113). Publish PRIVATE
`remyxai/panorama-flux-modular` → load via `trust_remote_code` → assert `PanoramaBlock` → a cheap
1536×768 few-step run → the **no-op spike**: one covering window must be **bit-exact stock FLUX** (the
scatter-average is the identity when a single window covers the canvas). Upload `block.py` first.
Runtime: A100 · `HUGGINGFACE_TOKEN` · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload `block.py` first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"PanoramaBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.PanoramaBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"PanoramaBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/panorama-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load

In [ ]:
from diffusers import ModularPipeline
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/panorama-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "PanoramaBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect PanoramaBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

## 4 · Milestone A — smoke (1536×768, few steps)

A cheap wide canvas: two window columns with overlap. Must return a wide image, no error.

In [ ]:
import torch
PROMPT = "a panoramic mountain range at golden hour, alpine lake in the foreground"
g = torch.Generator(DEV).manual_seed(0)
sm = pipe(prompt=PROMPT, height=768, width=1536, window=768, stride=384,
          num_inference_steps=8, generator=g).images[0]
assert sm.size == (1536, 768), sm.size
print("[SMOKE] ran; output size", sm.size)   # expect (1536, 768)
sm.save("smoke.png"); display(sm.resize((768, 384)))

## 5 · Milestone B — no-op spike (the fusion seam)

`window >= canvas` ⇒ exactly **one** window covers the canvas ⇒ the per-window `img_ids` are the stock
full-canvas ids and the scatter-**average** is the identity ⇒ the loop reduces to **stock FLUX**.
This de-risks the two things the brief flags (per-window img_ids, and the averaging math) before the
full e2e. Compared against a stock `FluxPipeline` in **latent** space (the HRDiT/DyPE bit-exact
convention — decoding would only add VAE noise to the comparison), same seed/prompt/steps/guidance.
The modular pipe is parked on CPU while stock runs, so only one FLUX copy is on the GPU at a time.

In [ ]:
import gc, torch, numpy as np
from diffusers import FluxPipeline

H = W = 512           # window == canvas -> exactly one covering window
STEPS, PROMPT = 4, "a red barn in a snowy field"

pipe.to("cpu"); gc.collect(); torch.cuda.empty_cache()      # park the modular pipe; one FLUX on GPU
stock = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=DT).to(DEV)
g2 = torch.Generator(DEV).manual_seed(0)
ref = stock(prompt=PROMPT, height=H, width=W, num_inference_steps=STEPS, guidance_scale=3.5,
            max_sequence_length=512, generator=g2, output_type="latent").images.float().cpu()
del stock; gc.collect(); torch.cuda.empty_cache()

pipe.to(DEV)                                               # bring the modular pipe back
g1 = torch.Generator(DEV).manual_seed(0)
ours = pipe(prompt=PROMPT, height=H, width=W, window=W, stride=W, num_inference_steps=STEPS,
            guidance_scale=3.5, max_sequence_length=512, generator=g1, output_type="latent").images[0].float().cpu()

# both sides return the PACKED latent (1, seq, C) — _pack_latents' layout
assert ours.shape == ref.shape, (ours.shape, ref.shape)
d = float((ours - ref).abs().max())
print(f"[NO-OP SPIKE] single-window vs stock FLUX (packed latents): max|Δ| = {d:.3e}")
print("[NO-OP SPIKE] PASS — one covering window == stock (ids + averaging are the identity)"
      if d < 5e-2 else "[NO-OP SPIKE] REVIEW — inspect the img_ids offsets / fusion buffers")

## Verdict

`loaded block: PanoramaBlock` + a wide 1536×768 image + the single-window spike matching stock FLUX
(max abs diff ~0) = the modular MultiDiffusion seam works. Then run `e2e.ipynb` for the full 3072×1024
panorama and the seam/replication check.